# SHINRA v2 S0 — Colab G4 rolling drum

**SHINRA is SHINRA. Created by NULLXES.**

Runtime: **G4 GPU** (RTX PRO 6000 Blackwell, SM120), Python 3, Latest.

| | |
|---|---|
| geometry | 2560 / 9728 / 36 / 32Q / 8KV / 128 · 3,969,056,256 tied |
| tokenizer | Unigram 131072, fetch files only |
| wrap | `[BOS=1] + body + [END_OF_TEXT=18]` |
| attention | **SDPA** (not FA2/FA3/FA4) |
| data | S0 20M, 100% synth |
| drum | produce N → SHA → train once → consume/delete → N+1 |
| hardware yaml | `configs/colab.yaml` (`max_tokens: 0`, `disk_ceiling_gb: 400`) |
| train recipe | `configs/stages/s0_bringup.yaml` |

Need `main` at **`90c5e9a`** or newer (`data/rolling_drum.py` must exist).

Forbidden: Hub weights, `from_pretrained` of a foreign checkpoint, identity QA, `chat_template.jinja`, `pip install .[flash]`, RTX 2080, pack 8192 as this path.

Crash: do not wipe `/content/shinra_scratch/s0`. Re-run the train cell. Resume reads `run/cursor.json` + last checkpoint SHA.

In [ ]:
import os
import torch

assert torch.cuda.is_available(), "need G4 GPU runtime"
name = torch.cuda.get_device_name(0)
print(name, torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")
assert "2080" not in name, "RTX 2080 is forbidden"

repo = "/content/NULLXES-SHINRA-4B-INSTRUCT"
if not os.path.exists(repo):
    !git clone https://github.com/MagistrTheOne/NULLXES-SHINRA-4B-INSTRUCT.git {repo}
%cd /content/NULLXES-SHINRA-4B-INSTRUCT
!git fetch origin
!git checkout main
!git pull origin main
!git log -1 --oneline
!pip install -q -e .
os.environ["PYTHONPATH"] = "/content/NULLXES-SHINRA-4B-INSTRUCT"

from model.modeling_shinra import ShinraPreTrainedModel
assert ShinraPreTrainedModel._supports_flash_attn_2 is False
assert ShinraPreTrainedModel._supports_sdpa is True
from pathlib import Path
assert Path("data/rolling_drum.py").is_file(), "rolling drum missing — git pull did not reach 90c5e9a+"

In [ ]:
!python scripts/v2_s0_colab.py --dry-hardware

## Tokenizer DNA only

Hub may still host **v1 weights**. This fetch allows tokenizer files only. No `*.safetensors`.

In [ ]:
!python scripts/fetch_tokenizer.py --dest tokenizer/artifacts
from pathlib import Path
arts = Path("tokenizer/artifacts")
weights = list(arts.rglob("*.safetensors")) + list(arts.rglob("*.bin"))
assert not weights, weights
print("tokenizer files", sorted(p.name for p in arts.iterdir() if p.is_file())[:12])

## S0 drum

`v2_s0_colab.py` → `v2_stage_run.py` → `data/rolling_drum.py`.

One active `.bin` at a time. Token budget is **>= 20M** after a completed optimizer step (`overshoot_tokens` in cursor).

Re-run this cell to resume. Do not `rm -rf /content/shinra_scratch/s0`.

OOM on first Adam step is the next gate (weights + grads + Adam + activations), not a reason to switch to flash-attn.

In [ ]:
!python scripts/v2_s0_colab.py --corpus-dir /content/shinra_scratch/corpus --output-dir /content/shinra_scratch/s0

## Status / cursor / ledger

In [ ]:
from pathlib import Path
import json

run = Path("/content/shinra_scratch/s0/run")
for name in ("status.json", "cursor.json"):
    p = run / name
    print("====", name, "====")
    print(p.read_text(encoding="utf-8") if p.exists() else "missing")

ledger = run / "ledger.jsonl"
if ledger.exists():
    events = [json.loads(line)["event"] for line in ledger.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("ledger events", events[:24], "..." if len(events) > 24 else "")
    print("n_events", len(events))
else:
    print("ledger missing")

metrics = run / "metrics.jsonl"
print("metrics lines", len(metrics.read_text(encoding="utf-8").splitlines()) if metrics.exists() else 0)

bins = sorted(Path("/content/shinra_scratch/corpus/s0").glob("*.bin")) if Path("/content/shinra_scratch/corpus/s0").exists() else []
print("active shard bins", [p.name for p in bins])